# 🤖 표준어 → 제주어 번역기 (Transformer 구조)

---

**논문 원본 구조 구현:**
```
Encoder: PositionalEmbedding → [MHA → Add&Norm → FFN → Add&Norm] × N
Decoder: PositionalEmbedding → [Masked MHA → Add&Norm → Cross MHA → Add&Norm → FFN → Add&Norm] × N
Output:  Dense(vocab_size) → Softmax
```


In [1]:
# =============================================================
# 1. 라이브러리 임포트 & 데이터 준비
# =============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ── 학습 데이터 ──────────────────────────────────────────────
# 표준어(입력) → 제주어(출력) 쌍
src_texts = ["어디 가세요", "밥 먹었어요", "지금 뭐해요", "날씨가 좋네요"]
tgt_texts = [
    "<sos> 어디 가쿠광 <eos>",
    "<sos> 밥 먹엉수다 <eos>",
    "<sos> 지금 뭐허는고 <eos>",
    "<sos> 날씨 좋수다 <eos>"
]

# ── 토큰화: 문장을 단어 단위로 분리 ──────────────────────────
# 예) "어디 가세요" → ["어디", "가세요"]
def tokenize(sentences):
    return [s.split() for s in sentences]

src_tokens = tokenize(src_texts)
tgt_tokens = tokenize(tgt_texts)

# ── 어휘 사전 생성 ────────────────────────────────────────────
# 학습 데이터에 등장한 모든 단어를 수집해 사전 구축
src_vocab = sorted({w for s in src_tokens for w in s})
tgt_vocab = sorted({w for s in tgt_tokens for w in s})

# StringLookup: 단어(문자열) → 정수 인덱스 변환기
src_lookup = tf.keras.layers.StringLookup(vocabulary=src_vocab, mask_token=None)
tgt_lookup = tf.keras.layers.StringLookup(vocabulary=tgt_vocab, mask_token=None)

src_vocab_size = len(src_lookup.get_vocabulary())  # 소스 어휘 수
tgt_vocab_size = len(tgt_lookup.get_vocabulary())  # 타겟 어휘 수

print(f"소스(표준어) 어휘 수: {src_vocab_size}")
print(f"타겟(제주어) 어휘 수: {tgt_vocab_size}")
print(f"소스 어휘: {src_lookup.get_vocabulary()}")
print(f"타겟 어휘: {tgt_lookup.get_vocabulary()}")

# ── 시퀀스 인코딩 & 패딩 ──────────────────────────────────────
# 모든 문장의 길이를 max_len으로 통일 (짧은 문장은 0으로 채움)
max_len = 10

def encode(lookup, tokens):
    ids = lookup(tokens)   # 단어 → 정수 인덱스
    # pad_sequences: 길이가 다른 시퀀스를 max_len으로 맞춤 (뒤를 0으로 채움)
    return tf.keras.preprocessing.sequence.pad_sequences(
        ids, maxlen=max_len, padding="post"
    )

# 인코더 입력: 표준어 문장 전체
encoder_input_data  = encode(src_lookup, src_tokens)

# 디코더 입력: <sos>부터 시작하는 제주어 (마지막 토큰 제외)
# Teacher Forcing: 학습 시 정답을 디코더 입력으로 직접 사용
decoder_input_data  = encode(tgt_lookup, [t[:-1] for t in tgt_tokens])

# 디코더 정답: <eos>로 끝나는 제주어 (첫 토큰 <sos> 제외)
# 모델이 예측해야 할 실제 정답 (한 칸 앞으로 shift된 구조)
decoder_target_data = encode(tgt_lookup, [t[1:]  for t in tgt_tokens])

print(f"\n인코더 입력 형태: {encoder_input_data.shape}")
print(f"디코더 입력 형태: {decoder_input_data.shape}")
print(f"디코더 정답 형태: {decoder_target_data.shape}")


C:\Users\fasoft\anaconda3\envs\TFWork\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.14) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


소스(표준어) 어휘 수: 9
타겟(제주어) 어휘 수: 11
소스 어휘: ['[UNK]', '가세요', '날씨가', '먹었어요', '뭐해요', '밥', '어디', '좋네요', '지금']
타겟 어휘: ['[UNK]', '<eos>', '<sos>', '가쿠광', '날씨', '먹엉수다', '뭐허는고', '밥', '어디', '좋수다', '지금']

인코더 입력 형태: (4, 10)
디코더 입력 형태: (4, 10)
디코더 정답 형태: (4, 10)


---
## 📍 Part 1. Positional Embedding (위치 임베딩)
트랜스포머는 단어를 **동시에** 처리하므로 순서 정보가 없습니다.  
→ **단어 임베딩 + 위치 임베딩** 을 더해 순서 정보를 추가합니다.


In [2]:
# =============================================================
# 2. Positional Embedding (위치 임베딩 레이어)
# =============================================================
# 역할: 단어 의미(token embedding) + 위치 정보(position embedding)를 결합
# RNN은 순서대로 처리해 자동으로 순서를 알지만,
# 트랜스포머는 동시에 처리하므로 위치 정보를 별도로 추가해야 합니다.

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        # 단어 임베딩: 단어 인덱스 → 의미 벡터
        # 예) '가세요'(인덱스 3) → [0.2, -0.1, 0.8, ...] (embed_dim차원)
        self.token_embeddings    = layers.Embedding(vocab_size,       embed_dim)
        
        # 위치 임베딩: 위치 번호 → 위치 벡터
        # 예) 0번째 단어 → [0.5, 0.3, ...], 1번째 단어 → [다른값, ...]
        self.position_embeddings = layers.Embedding(sequence_length,  embed_dim)

    def call(self, inputs):
        length = tf.shape(inputs)[-1]   # 현재 시퀀스 길이
        
        # 0, 1, 2, ... 위치 번호 생성
        positions = tf.range(start=0, limit=length, delta=1)
        
        # 단어 의미 벡터
        embedded_tokens    = self.token_embeddings(inputs)
        
        # 위치 정보 벡터
        embedded_positions = self.position_embeddings(positions)
        
        # 두 벡터를 더해 "단어 의미 + 위치 정보"를 하나의 벡터로 표현
        return embedded_tokens + embedded_positions

print("✅ PositionalEmbedding 레이어 정의 완료")
print("   단어 임베딩(의미) + 위치 임베딩(순서) → 합산")


✅ PositionalEmbedding 레이어 정의 완료
   단어 임베딩(의미) + 위치 임베딩(순서) → 합산


---
## 🔵 Part 2. Transformer Encoder Layer (인코더 레이어)

```
입력 X
 ↓ Multi-Head Self-Attention   (문장 내 단어들의 관계 파악)
 ↓ Add & Norm                  (잔차 연결 + 정규화)
 ↓ Feed-Forward Network        (각 단어의 특징 변환)
 ↓ Add & Norm                  (잔차 연결 + 정규화)
출력
```


In [3]:
# =============================================================
# 3. Transformer Encoder Layer (인코더 레이어 1개)
# =============================================================
# 인코더: 표준어 문장 전체를 읽고 문맥 정보를 추출
# 핵심 구조: Self-Attention → Add&Norm → FFN → Add&Norm

class TransformerEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        """
        매개변수:
          embed_dim    : 임베딩(벡터) 차원 수
          num_heads    : Multi-Head Attention 헤드 수
                         (여러 관점에서 동시에 Attention 계산)
          ff_dim       : FFN 내부 차원 (보통 embed_dim × 4)
          dropout_rate : 학습 시 무작위로 뉴런을 끄는 비율 (과적합 방지)
        """
        super().__init__()

        # ── ① Multi-Head Self-Attention ──────────────────────
        # "각 단어가 문장 내 다른 단어들과 얼마나 관련있나?" 를 num_heads개 관점에서 동시 계산
        # key_dim: 각 헤드 내부 차원 (embed_dim // num_heads 권장)
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        # ── ② Feed-Forward Network (FFN) ─────────────────────
        # Attention이 "단어 간 관계"를 학습한다면,
        # FFN은 "각 단어의 특징"을 더 풍부하게 변환하는 역할
        # 구조: Linear(확장) → ReLU(비선형) → Linear(복원)
        self.ffn = keras.Sequential([
            # d_model → ff_dim: 차원을 크게 확장 (더 많은 특징 추출)
            layers.Dense(ff_dim, activation="relu"),
            # ff_dim → embed_dim: 다시 원래 차원으로 복원
            layers.Dense(embed_dim),
        ])

        # ── ③ Layer Normalization (레이어 정규화) × 2 ────────
        # 값의 크기가 너무 커지거나 작아지지 않도록 균일하게 정규화
        # → 학습 안정화 효과
        self.layernorm_1 = layers.LayerNormalization(epsilon=1e-6)  # MHA 뒤
        self.layernorm_2 = layers.LayerNormalization(epsilon=1e-6)  # FFN 뒤

        # ── ④ Dropout ─────────────────────────────────────────
        # 학습 시 일부 뉴런을 무작위로 비활성화 → 과적합 방지
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.dropout_2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        """
        인코더 레이어 순전파 (Forward Pass)
        inputs  : (batch_size, seq_len, embed_dim) 입력 행렬
        training: True=학습모드(Dropout 적용), False=추론모드
        """
        # ════════════════════════════════════════════════
        # [블록 1] Multi-Head Self-Attention + Add & Norm
        # ════════════════════════════════════════════════
        # Self-Attention: query=key=value=inputs (자기 자신에게 Attention)
        # → 같은 문장의 단어들끼리 서로의 관계를 파악
        attn_output = self.attention(
            query=inputs,   # Q: "나는 무엇을 찾고 있나?"
            key=inputs,     # K: "각 단어는 어떤 정보를 갖고 있나?"
            value=inputs    # V: "실제로 전달할 정보"
        )
        attn_output = self.dropout_1(attn_output, training=training)

        # Add & Norm: 원본 입력(inputs)과 Attention 출력을 더한 후 정규화
        # 잔차 연결(inputs + attn_output): 원본 정보를 잃지 않도록 보존
        out_1 = self.layernorm_1(inputs + attn_output)

        # ════════════════════════════════════════════════
        # [블록 2] Feed-Forward Network + Add & Norm
        # ════════════════════════════════════════════════
        # FFN: 각 단어(토큰)에 독립적으로 적용 (단어 특징 변환)
        ffn_output = self.ffn(out_1)
        ffn_output = self.dropout_2(ffn_output, training=training)

        # Add & Norm: FFN 입력(out_1)과 FFN 출력을 더한 후 정규화
        out_2 = self.layernorm_2(out_1 + ffn_output)

        return out_2   # (batch_size, seq_len, embed_dim) 형태 유지

print("✅ TransformerEncoderLayer 정의 완료")
print("   구조: Self-Attention → Add&Norm → FFN → Add&Norm")


✅ TransformerEncoderLayer 정의 완료
   구조: Self-Attention → Add&Norm → FFN → Add&Norm


---
## 🟠 Part 3. Transformer Decoder Layer (디코더 레이어)

```
입력 Y (이전까지 생성된 번역 토큰)
 ↓ Masked Multi-Head Self-Attention  (미래 단어 차단, 이전 생성 참조)
 ↓ Add & Norm
 ↓ Cross-Attention                   (인코더 출력 참조 ← 핵심!)
 ↓ Add & Norm
 ↓ Feed-Forward Network
 ↓ Add & Norm
출력
```


In [4]:
# =============================================================
# 4. Transformer Decoder Layer (디코더 레이어 1개)
# =============================================================
# 디코더: 인코더의 문맥 정보를 참고하면서 제주어를 한 단어씩 생성
# 인코더와 다른 점:
#   ① Masked Self-Attention (미래 단어 컨닝 금지)
#   ② Cross-Attention       (인코더 출력 참조)

class TransformerDecoderLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()

        # ── ① Masked Multi-Head Self-Attention ───────────────
        # 디코더가 이미 생성한 단어들끼리의 관계 파악
        # use_causal_mask=True: 미래 단어를 참조하지 못하도록 마스킹
        # 예) '밥'을 생성할 때, 아직 생성 안 된 '먹엉수다'는 볼 수 없음
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        # ── ② Cross-Attention ─────────────────────────────────
        # 디코더(번역 중인 단어)가 인코더(원문 단어들)를 참조
        # Q → 디코더에서 (내가 지금 생성하려는 단어)
        # K, V → 인코더에서 (원문의 모든 단어 정보)
        # 예) '먹엉수다' 생성 시 → 원문 '먹었어요'에 집중
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        # ── ③ Feed-Forward Network (인코더와 동일 구조) ──────
        # 각 토큰의 특징을 더 복잡하게 변환
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),  # 차원 확장 + 비선형 변환
            layers.Dense(embed_dim),                  # 차원 복원
        ])

        # ── ④ Layer Normalization × 3 ─────────────────────────
        # 디코더는 서브레이어가 3개이므로 LayerNorm도 3개 필요
        self.layernorm_1 = layers.LayerNormalization(epsilon=1e-6)  # Masked MHA 뒤
        self.layernorm_2 = layers.LayerNormalization(epsilon=1e-6)  # Cross-Attn 뒤
        self.layernorm_3 = layers.LayerNormalization(epsilon=1e-6)  # FFN 뒤

        # ── ⑤ Dropout × 3 ────────────────────────────────────
        self.dropout_1 = layers.Dropout(dropout_rate)
        self.dropout_2 = layers.Dropout(dropout_rate)
        self.dropout_3 = layers.Dropout(dropout_rate)

    def call(self, inputs, encoder_outputs, training=False):
        """
        디코더 레이어 순전파
        inputs          : 디코더 입력 (batch, tgt_seq, embed_dim)
        encoder_outputs : 인코더 최종 출력 = Context (batch, src_seq, embed_dim)
        training        : 학습/추론 모드 구분
        """
        # ════════════════════════════════════════════════
        # [블록 1] Masked Self-Attention + Add & Norm
        # ════════════════════════════════════════════════
        # use_causal_mask=True: Keras가 자동으로 상삼각 마스크 생성
        # → 현재 위치 이후의 단어(미래)에는 Attention이 0이 됨
        # → 학습 시에도 실제 추론과 동일한 조건을 유지
        attn_output_1 = self.attention_1(
            query=inputs,
            key=inputs,
            value=inputs,
            use_causal_mask=True    # ← 미래 단어 차단 핵심!
        )
        attn_output_1 = self.dropout_1(attn_output_1, training=training)
        # 잔차 연결 + 정규화
        out_1 = self.layernorm_1(inputs + attn_output_1)

        # ════════════════════════════════════════════════
        # [블록 2] Cross-Attention + Add & Norm
        # ════════════════════════════════════════════════
        # Q = 디코더(out_1): "내가 지금 어떤 단어를 생성하려 하는가?"
        # K = 인코더(encoder_outputs): "원문에서 어떤 단어들이 있나?"
        # V = 인코더(encoder_outputs): "원문 단어들의 실제 정보"
        # → 번역 중인 단어가 원문의 어느 단어를 참고할지 결정
        attn_output_2 = self.attention_2(
            query=out_1,              # 디코더에서
            key=encoder_outputs,      # 인코더에서
            value=encoder_outputs     # 인코더에서
        )
        attn_output_2 = self.dropout_2(attn_output_2, training=training)
        # 잔차 연결 + 정규화
        out_2 = self.layernorm_2(out_1 + attn_output_2)

        # ════════════════════════════════════════════════
        # [블록 3] Feed-Forward Network + Add & Norm
        # ════════════════════════════════════════════════
        # 각 위치의 토큰에 독립적으로 FFN 적용
        ffn_output = self.ffn(out_2)
        ffn_output = self.dropout_3(ffn_output, training=training)
        # 잔차 연결 + 정규화
        out_3 = self.layernorm_3(out_2 + ffn_output)

        return out_3   # (batch, tgt_seq, embed_dim)

print("✅ TransformerDecoderLayer 정의 완료")
print("   구조: Masked Self-Attn → Add&Norm")
print("         Cross-Attention  → Add&Norm")
print("         FFN              → Add&Norm")


✅ TransformerDecoderLayer 정의 완료
   구조: Masked Self-Attn → Add&Norm
         Cross-Attention  → Add&Norm
         FFN              → Add&Norm


---
## 🏗️ Part 4. 완성된 Transformer 모델 조립

인코더 레이어 N개 + 디코더 레이어 N개를 쌓아 완성합니다.


In [5]:
# =============================================================
# 5. 완성된 Transformer 모델 (Keras Functional API)
# =============================================================
# 하이퍼파라미터: 모델 구조를 결정하는 설정값
embed_dim    = 128   # 임베딩 벡터 차원 (실제 논문: 512)
num_heads    = 4     # Attention 헤드 수 (실제 논문: 8)
ff_dim       = 512   # FFN 내부 차원 (보통 embed_dim × 4 = 128×4)
num_layers   = 2     # 인코더/디코더 레이어 반복 수 (실제 논문: 6)
dropout_rate = 0.1   # Dropout 비율 10%

# ══════════════════════════════════════════════════════════
# [인코더 블록]
# 표준어 문장 전체를 읽어 문맥 정보(Context)를 생성
# ══════════════════════════════════════════════════════════

# 인코더 입력: 표준어 토큰 인덱스 시퀀스
encoder_inputs = layers.Input(shape=(None,), name="encoder_input")

# 위치 임베딩: 단어 의미 + 위치 정보
x = PositionalEmbedding(
    sequence_length=max_len,
    vocab_size=src_vocab_size,
    embed_dim=embed_dim
)(encoder_inputs)

# 인코더 레이어를 num_layers번 반복해서 쌓기
# 층을 깊게 쌓을수록 더 복잡한 문맥 정보를 학습
for _ in range(num_layers):
    x = TransformerEncoderLayer(
        embed_dim=embed_dim,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout_rate=dropout_rate
    )(x)

# x = 인코더 최종 출력 (Context 행렬)
# 형태: (batch_size, src_seq_len, embed_dim)
# 의미: 표준어 각 단어의 문맥을 담은 벡터들
encoder_outputs = x

# ══════════════════════════════════════════════════════════
# [디코더 블록]
# 인코더의 Context를 참고하면서 제주어를 한 단어씩 생성
# ══════════════════════════════════════════════════════════

# 디코더 입력: 제주어 토큰 인덱스 시퀀스 (<sos>로 시작)
decoder_inputs = layers.Input(shape=(None,), name="decoder_input")

# 위치 임베딩 (타겟 어휘 사전 사용)
y = PositionalEmbedding(
    sequence_length=max_len,
    vocab_size=tgt_vocab_size,
    embed_dim=embed_dim
)(decoder_inputs)

# 디코더 레이어를 num_layers번 반복
# 각 레이어에 인코더 출력(encoder_outputs)을 전달
for _ in range(num_layers):
    y = TransformerDecoderLayer(
        embed_dim=embed_dim,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout_rate=dropout_rate
    )(y, encoder_outputs)   # ← 인코더 Context 전달!

# ══════════════════════════════════════════════════════════
# [출력 레이어]
# 각 위치에서 다음 단어의 확률 분포를 계산
# ══════════════════════════════════════════════════════════
# Dense: embed_dim → tgt_vocab_size (어휘 수만큼 차원 확장)
# softmax: 각 단어의 확률값으로 변환 (합=1.0)
outputs = layers.Dense(tgt_vocab_size, activation="softmax", name="output")(y)

# ══════════════════════════════════════════════════════════
# 모델 조립 & 컴파일
# ══════════════════════════════════════════════════════════
transformer = keras.Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=outputs,
    name="Transformer"
)

# 컴파일: 학습 방법 설정
transformer.compile(
    # Adam: 학습률을 자동으로 조절하는 최적화기
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    # SparseCategoricalCrossentropy: 정수 레이블용 교차 엔트로피 손실
    # (원-핫 인코딩 없이 정수 인덱스로 바로 사용 가능)
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 모델 구조 요약 출력
transformer.summary()
print(f"\n✅ 완전한 Transformer 모델 생성 완료!")
print(f"   인코더 레이어: {num_layers}개 × (Self-Attn → Add&Norm → FFN → Add&Norm)")
print(f"   디코더 레이어: {num_layers}개 × (Masked Attn → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm)")


Model: "Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)    │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ positional_embedding          │ (None, None, 128)         │           2,432 │ encoder_input[0][0]        │
│ (PositionalEmbedding)         │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_input (InputLayer)    │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ transformer_encoder_layer     │ (None, None, 128)         │         198,272 │ positional_embedding[0][0] │
│ (TransformerEncoderLayer)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ positional_embedding_1        │ (None, None, 128)         │           2,688 │ decoder_input[0][0]        │
│ (PositionalEmbedding)         │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ transformer_encoder_layer_1   │ (None, None, 128)         │         198,272 │ transformer_encoder_layer… │
│ (TransformerEncoderLayer)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ transformer_decoder_layer     │ (None, None, 128)         │         264,576 │ positional_embedding_1[0]… │
│ (TransformerDecoderLayer)     │                           │                 │ transformer_encoder_layer… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ transformer_decoder_layer_1   │ (None, None, 128)         │         264,576 │ transformer_decoder_layer… │
│ (TransformerDecoderLayer)     │                           │                 │ transformer_encoder_layer… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output (Dense)                │ (None, None, 11)          │           1,419 │ transformer_decoder_layer… │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 932,235 (3.56 MB)

 Trainable params: 932,235 (3.56 MB)

 Non-trainable params: 0 (0.00 B)


✅ 완전한 Transformer 모델 생성 완료!
   인코더 레이어: 2개 × (Self-Attn → Add&Norm → FFN → Add&Norm)
   디코더 레이어: 2개 × (Masked Attn → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm)


---
## 🏋️ Part 5. 모델 학습


In [6]:
# =============================================================
# 6. 모델 학습
# =============================================================
# epochs: 전체 데이터를 몇 번 반복 학습할지
# batch_size: 한 번에 몇 개의 데이터를 처리할지
# verbose=1: 학습 진행 상황 출력

history = transformer.fit(
    x=[encoder_input_data, decoder_input_data],  # 인코더 입력, 디코더 입력
    y=decoder_target_data,                        # 정답 레이블
    epochs=500,       # 소규모 데이터이므로 많이 반복
    batch_size=2,     # 전체 4개 데이터를 2개씩 나누어 처리
    verbose=1
)

# 최종 정확도 출력
final_loss     = history.history['loss'][-1]
final_accuracy = history.history['accuracy'][-1]
print(f"\n✅ 학습 완료!")
print(f"   최종 Loss    : {final_loss:.4f}")
print(f"   최종 Accuracy: {final_accuracy:.4f}")


Epoch 1/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 20s 53ms/step - accuracy: 0.5500 - loss: 1.5384
Epoch 2/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.8500 - loss: 0.4514
Epoch 3/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.9000 - loss: 0.2798 
Epoch 4/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.9250 - loss: 0.2635
Epoch 5/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9250 - loss: 0.2084
Epoch 6/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.9500 - loss: 0.1677
Epoch 7/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.9250 - loss: 0.1861
Epoch 8/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.9250 - loss: 0.1728 
Epoch 9/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.9000 - loss: 0.1421 
Epoch 10/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.9000 - loss: 0.1645
Epoch 11/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9250 - loss: 0.1318
Epoch 12/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9500 

---
## 🔄 Part 6. 번역 추론 (Autoregressive 생성)

학습된 모델로 실제 번역을 수행합니다.  
**이전 출력을 다음 입력으로 사용**하는 자기회귀(Autoregressive) 방식입니다.

```
Step 1: 인코더에 표준어 문장 입력 → Context 생성 (한 번만!)
Step 2: 디코더에 [<sos>] 입력     → 첫 단어 예측
Step 3: 디코더에 [<sos>, 예측1]   → 두 번째 단어 예측
Step 4: <eos> 가 나올 때까지 반복
```


In [7]:
# =============================================================
# 7. 번역 추론 함수 (Autoregressive Decoding)
# =============================================================
# 타겟 어휘 인덱스 → 단어 변환용 사전
idx_to_word = tgt_lookup.get_vocabulary()

def translate(sentence, verbose=False):
    """
    표준어 문장을 제주어로 번역하는 함수
    
    매개변수:
      sentence : 번역할 표준어 문장 (문자열)
      verbose  : True면 각 스텝의 예측 과정을 출력
    반환값:
      번역된 제주어 문장 (문자열)
    """
    # ── ① 입력 문장 인코딩 ───────────────────────────────────
    tokens    = sentence.split()
    enc_input = encode(src_lookup, [tokens])   # (1, max_len)

    # ── ② <sos> 토큰으로 디코딩 시작 ────────────────────────
    # Autoregressive: 처음엔 <sos>만 갖고 시작
    decoded_sentence = ["<sos>"]

    # ── ③ 단어를 한 개씩 생성 ────────────────────────────────
    for step in range(max_len - 1):
        # 현재까지 생성된 시퀀스를 디코더 입력으로 사용
        dec_input = encode(tgt_lookup, [decoded_sentence])   # (1, max_len)

        # 모델 예측: 각 위치에서 다음 단어 확률 분포
        # predictions 형태: (1, tgt_seq_len, tgt_vocab_size)
        predictions = transformer.predict(
            [enc_input, dec_input], verbose=0
        )

        # 현재 생성 위치(step)에서 가장 확률이 높은 단어 선택 (Greedy Decoding)
        predicted_id   = np.argmax(predictions[0, step, :])
        next_word      = idx_to_word[predicted_id]

        if verbose:
            top3 = np.argsort(predictions[0, step, :])[::-1][:3]
            top3_words = [(idx_to_word[i], f"{predictions[0,step,i]:.3f}") for i in top3]
            print(f"  Step {step+1}: {decoded_sentence} → 예측={next_word} | 상위3={top3_words}")

        # <eos> 가 예측되면 생성 완료
        if next_word == "<eos>":
            break

        decoded_sentence.append(next_word)

    # <sos> 제외하고 반환
    return " ".join(decoded_sentence[1:])


# =============================================================
# 8. 학습 데이터 번역 테스트
# =============================================================
print("=" * 50)
print("📝 학습 데이터 번역 결과")
print("=" * 50)
test_sentences = ["어디 가세요", "밥 먹었어요", "지금 뭐해요", "날씨가 좋네요"]
answers        = ["어디 가쿠광", "밥 먹엉수다", "지금 뭐허는고", "날씨 좋수다"]

for src, ans in zip(test_sentences, answers):
    result = translate(src)
    match  = "✅" if result == ans else "❌"
    print(f"  {match} {src} → {result}  (정답: {ans})")

print()
print("=" * 50)
print("🆕 새로운 문장 번역 테스트 (학습 데이터 조합)")
print("=" * 50)
new_sentences = ["지금 가세요", "지금 좋네요", "어디 먹었어요"]
for src in new_sentences:
    result = translate(src)
    print(f"  {src} → {result}")


📝 학습 데이터 번역 결과
  ✅ 어디 가세요 → 어디 가쿠광  (정답: 어디 가쿠광)
  ✅ 밥 먹었어요 → 밥 먹엉수다  (정답: 밥 먹엉수다)
  ✅ 지금 뭐해요 → 지금 뭐허는고  (정답: 지금 뭐허는고)
  ✅ 날씨가 좋네요 → 날씨 좋수다  (정답: 날씨 좋수다)

🆕 새로운 문장 번역 테스트 (학습 데이터 조합)
  지금 가세요 → 어디 가쿠광
  지금 좋네요 → 날씨 좋수다
  어디 먹었어요 → 어디 가쿠광


---
## 🔍 Part 7. 번역 과정 상세 확인


In [8]:
# =============================================================
# 9. 번역 과정 단계별 상세 출력
# =============================================================
print("🔍 '밥 먹었어요' 번역 과정 상세 보기")
print("=" * 55)
result = translate("밥 먹었어요", verbose=True)
print(f"\n최종 번역: 밥 먹었어요 → {result}")


🔍 '밥 먹었어요' 번역 과정 상세 보기
  Step 1: ['<sos>'] → 예측=밥 | 상위3=[('밥', '1.000'), ('날씨', '0.000'), ('어디', '0.000')]
  Step 2: ['<sos>', '밥'] → 예측=먹엉수다 | 상위3=[('먹엉수다', '1.000'), ('뭐허는고', '0.000'), ('가쿠광', '0.000')]
  Step 3: ['<sos>', '밥', '먹엉수다'] → 예측=<eos> | 상위3=[('<eos>', '1.000'), ('뭐허는고', '0.000'), ('<sos>', '0.000')]

최종 번역: 밥 먹었어요 → 밥 먹엉수다


---
## 📋 전체 구조 요약

```
표준어 입력
    ↓
[인코더] × 2 레이어
    PositionalEmbedding (단어 의미 + 위치 정보)
    ① Multi-Head Self-Attention  → Add & Norm
    ② Feed-Forward Network       → Add & Norm
    ↓
Context 행렬 (표준어 문맥 정보)
    ↓
[디코더] × 2 레이어       ← Context 참조
    PositionalEmbedding
    ① Masked Self-Attention      → Add & Norm  (미래 차단)
    ② Cross-Attention            → Add & Norm  (표준어 참조)
    ③ Feed-Forward Network       → Add & Norm
    ↓
Dense(vocab_size) + Softmax
    ↓
제주어 출력 (단어별 확률 분포)
```

| 구성요소 | 완성된 코드 |
|---------|------------|
| 인코더 Self-Attention + Add&Norm | ✅ |
| **인코더 FFN + Add&Norm** | ✅ |
| 디코더 Masked Self-Attention + Add&Norm | ✅ |
| 디코더 Cross-Attention + Add&Norm | ✅ |
| **디코더 FFN + Add&Norm** | ✅ |
| **Dropout (과적합 방지)** | ✅ |
| **N개 레이어 반복** | ✅ num_layers개 |

전 학습이 없더라도 적은 데이터 내에서 단어 활용의 원리를 더 빠르게 터득합니다.